# 国会議事録 API ハンズオン（Section 06）

このノートブックでは、国会議事録 API からデータを取得し、
Python で確認・加工していく流れを体験します。

---

## ここでやること
- API からデータを取得する
- 取得パラメータを変えて結果の違いを見る
- 表形式に整形して中身を確認する
- CSV に保存して、他のツールでも使える形にする

> コードは用意されています。  
> **基本は「実行するだけ」でOK**。安心して進めてください。

In [ ]:
import requests

# 国会議事録 API のエンドポイント
BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

# まずはシンプルな条件で少しだけデータを取ってみる
params = {
    "recordPacking": "json",   # JSON形式で受け取る指定
    "any": "教育",             # キーワード（とりあえず「教育」で検索）
    "from": "2020-01-01",      # 検索開始日
    "until": "2025-12-31",     # 検索終了日
    "maximumRecords": 10       # 取得件数（まずは10件だけ）
}

# APIにリクエストを送信
response = requests.get(BASE_URL, params=params)

# エラーがあればここで例外を出す（デバッグしやすくするため）
response.raise_for_status()

# JSONデータとして読み込む
data = response.json()

# 国会議事録APIは、"speechRecord" というキーに発言データが入っている想定
records = data.get("speechRecord", [])

# 先頭数件だけ表示
for i, rec in enumerate(records[:5], start=1):
    print("--------------------------------")
    print(f"[{i}] 日付     : {rec.get('date')}")
    print(f"    委員会   : {rec.get('nameOfMeeting')}")
    print(f"    発言者   : {rec.get('speaker')}")
    text = rec.get("speech", "")
    print(f"    発言冒頭 : {text[:60]}{'...' if len(text) > 60 else ''}")


In [ ]:
# =========================================
# 初期セットアップ
# （外部ファイルを自動生成します）
# =========================================

import os

if not os.path.exists("output_layouts.py"):
    from textwrap import dedent

    with open("output_layouts.py", "w", encoding="utf-8") as f:
        f.write(dedent("""
        def show_card_layout(records):
            print("=== カード形式の出力例 ===")
            for i, rec in enumerate(records[:3], start=1):
                print("-" * 40)
                print(f"[{i}件目]")
                print(f"発言ID      : {rec.get('speechID')}")
                print(f"日付        : {rec.get('date')}")
                print(f"会議名      : {rec.get('nameOfMeeting')}")
                print(f"発言者      : {rec.get('speaker')}")
                speech = (rec.get("speech") or "")
                print("\\n【発言内容（冒頭）】")
                print(speech[:60] + ("..." if len(speech) > 60 else ""))


        def show_summary_layout(records):
            print("=== 1行サマリー形式の出力例 ===")
            for i, rec in enumerate(records[:3], start=1):
                speech = (rec.get("speech") or "")
                head = speech[:40] + ("..." if len(speech) > 40 else "")
                print(f"[{i}] {rec.get('date')} / {rec.get('nameOfMeeting')} / {rec.get('speaker')}")
                print(f"    → {head}")


        def show_table_layout(records):
            print("=== 表形式の出力例 ===")
            print("No | 日付        | 委員会名         | 発言者     | 発言冒頭")
            print("-" * 80)
            for i, rec in enumerate(records[:3], start=1):
                speech = (rec.get("speech") or "")
                head = speech[:20] + ("..." if len(speech) > 20 else "")
                print(f"{i:<2} | {rec.get('date',''):<10} | {rec.get('nameOfMeeting','')[:12]:<12} | "
                      f"{rec.get('speaker','')[:8]:<8} | {head}")
        """))

    print("output_layouts.py を生成しました")
else:
    print("output_layouts.py はすでに存在しています。")


In [ ]:
import requests
import output_layouts

# 国会議事録 API のエンドポイント
BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

# まずはシンプルな条件で少しだけデータを取ってみる
params = {
    "recordPacking": "json",   # JSON形式で受け取る指定
    "any": "教育",             # キーワード（とりあえず「教育」で検索）
    "from": "2020-01-01",      # 検索開始日
    "until": "2025-12-31",     # 検索終了日
    "maximumRecords": 10       # 取得件数（まずは10件だけ）
}

# APIにリクエストを送信
response = requests.get(BASE_URL, params=params)

# エラーがあればここで例外を出す（デバッグしやすくするため）
response.raise_for_status()

# JSONデータとして読み込む
data = response.json()

# 国会議事録APIは、"speechRecord" というキーに発言データが入っている想定
records = data.get("speechRecord", [])

output_layouts.show_card_layout(records)

In [ ]:
# =========================================
# 会議情報の取得
# =========================================
import requests

BASE_URL = "https://kokkai.ndl.go.jp/api/meeting"

params = {
    "recordPacking": "json",     # JSON形式で受け取る
    "nameOfMeeting": "文教科学委員会 文部科学委員会", # 会議名で指定
    "from": "2025-04-01",        # 検索開始日
    "until": "2025-04-30",       # 検索終了日
    "maximumRecords": 10         # 取得件数
}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()

data = response.json()
records = data.get("meetingRecord", [])

for i, rec in enumerate(records, start=1):
    print("--------------------------------")
    print(f"[{i}] 日付     : {rec.get('date')}")
    print(f"    院名     : {rec.get('nameOfHouse')}")
    print(f"    委員会   : {rec.get('nameOfMeeting')}")
    print(f"    会議ID   : {rec.get('issueID')}")
    speech_count = len(rec.get("speechRecord", []))
    print(f"    発言数   :{speech_count} 件")


In [ ]:
# =========================================
# 会議情報から発言情報を取得
# =========================================
import requests
import time

# meeting API（会議取得）
MEETING_URL = "https://kokkai.ndl.go.jp/api/meeting"

# speech API（発言取得）
SPEECH_URL = "https://kokkai.ndl.go.jp/api/speech"

# -----------------------------
# ① まず会議一覧を取得
# -----------------------------
meeting_params = {
    "recordPacking": "json",
    "nameOfMeeting": "文教科学委員会 文部科学委員会",
    "from": "2025-04-01",
    "until": "2025-04-30",
    "maximumRecords": 10
}

response = requests.get(MEETING_URL, params=meeting_params)
response.raise_for_status()

data = response.json()
meeting_records = data.get("meetingRecord", [])

print(f"対象会議数: {len(meeting_records)} 件")

# -----------------------------
# ② 各会議IDごとに speech取得
# -----------------------------
all_speeches = []
total_count = 0

for m_idx, meeting_rec in enumerate(meeting_records, start=1):

    issue_id = meeting_rec.get("issueID")
    meeting_name = meeting_rec.get("nameOfMeeting")
    meeting_date = meeting_rec.get("date")

    print("--------------------------------")
    print(f"[{m_idx}] {meeting_date} / {meeting_name}")
    print(f"会議ID: {issue_id}")

    start_record = 1

    while True:

        speech_params = {
            "recordPacking": "json",
            "issueID": issue_id,
            "maximumRecords": 100,
            "startRecord": start_record
        }

        speech_response = requests.get(SPEECH_URL,
                                       params=speech_params)
        speech_response.raise_for_status()

        speech_data = speech_response.json()
        speech_records = speech_data.get("speechRecord", [])

        if not speech_records:
            break

        all_speeches.extend(speech_records)
        total_count += len(speech_records)

        # 100件ごとに進捗表示
        print(f"  発言取得中... 合計 {total_count} 件")

        # 次ページへ
        start_record += len(speech_records)

        # 100件未満なら最後
        if len(speech_records) < 100:
            break

        time.sleep(0.3)

print("--------------------------------")
print(f"全発言取得完了: {total_count} 件")

In [ ]:
# =========================================
# CSV作成➡ダウンロード
# =========================================
import requests
import time
import pandas as pd
from google.colab import files

# meeting API（会議取得）
MEETING_URL = "https://kokkai.ndl.go.jp/api/meeting"

# speech API（発言取得）
SPEECH_URL = "https://kokkai.ndl.go.jp/api/speech"

# -----------------------------
# ① まず会議一覧を取得
# -----------------------------
meeting_params = {
    "recordPacking": "json",
    "nameOfMeeting": "文教科学委員会 文部科学委員会",
    "from": "2025-04-01",
    "until": "2025-04-30",
    "maximumRecords": 10
}

response = requests.get(MEETING_URL, params=meeting_params)
response.raise_for_status()

data = response.json()
meeting_records = data.get("meetingRecord", [])

print(f"対象会議数: {len(meeting_records)} 件")

# -----------------------------
# ② 各会議IDごとに speech取得
# -----------------------------
all_speeches = []
total_count = 0

for m_idx, meeting_rec in enumerate(meeting_records, start=1):

    issue_id = meeting_rec.get("issueID")
    meeting_name = meeting_rec.get("nameOfMeeting")
    meeting_date = meeting_rec.get("date")

    print("--------------------------------")
    print(f"[{m_idx}] {meeting_date} / {meeting_name}")
    print(f"会議ID: {issue_id}")

    start_record = 1

    while True:

        speech_params = {
            "recordPacking": "json",
            "issueID": issue_id,
            "maximumRecords": 100,
            "startRecord": start_record
        }

        speech_response = requests.get(SPEECH_URL,
                                       params=speech_params)
        speech_response.raise_for_status()

        speech_data = speech_response.json()
        speech_records = speech_data.get("speechRecord", [])

        if not speech_records:
            break

        # 会議情報を付与して保存
        for row in speech_records:
            row["meeting_name"] = meeting_name
            row["meeting_date"] = meeting_date
            row["issue_id"] = issue_id

        all_speeches.extend(speech_records)
        total_count += len(speech_records)

        # 進捗表示
        print(f"  発言取得中... 合計 {total_count} 件")

        # 次ページへ
        start_record += len(speech_records)

        # 100件未満なら最後
        if len(speech_records) < 100:
            break

        time.sleep(0.3)

print("--------------------------------")
print(f"全発言取得完了: {total_count} 件")

# -----------------------------
# ③ DataFrame化
# -----------------------------
df = pd.DataFrame(all_speeches)

# 欲しい列だけ並べ替え（存在する列のみ）
use_cols = [
    "meeting_date",
    "meeting_name",
    "issue_id",
    "speechID",
    "speechOrder",
    "speaker",
    "speakerPosition",
    "speakerGroup",
    "speech"
]

df = df[[c for c in use_cols if c in df.columns]]

# -----------------------------
# ④ CSV保存
# -----------------------------
file_name = "kokkai_speech_2025_04.csv"

df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV保存完了: {file_name}")

# -----------------------------
# ⑤ ダウンロード
# -----------------------------
files.download(file_name)
